In [0]:
!pip install torch torchvision

In [0]:
import os
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Subset

import mlflow
import mlflow.pyfunc
from mlflow import MlflowClient
from sklearn.metrics import confusion_matrix, classification_report

warnings.filterwarnings("ignore", category=FutureWarning)

# -------------------------
# Config
# -------------------------
DATA_ROOT = "/Volumes/tomato_data/default/raw/tomato/"
TEST_DIR  = os.path.join(DATA_ROOT, "test")

MODEL_NAME        = "workspace.default.tomato_disease_classifier"
CHAMPION_ALIAS    = "champion"
MODEL_URI         = f"models:/{MODEL_NAME}@{CHAMPION_ALIAS}"

IMAGE_SIZE        = 224
BATCH_SIZE        = 32
NUM_WORKERS       = 2
EVAL_SUBSET_SIZE  = 200   # subset of test set for the accuracy check

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CLASS_DISPLAY_NAMES = {
    "Tomato___Bacterial_spot": "Bacterial Spot",
    "Tomato___Early_blight": "Early Blight",
    "Tomato___Late_blight": "Late Blight",
    "Tomato___Leaf_Mold": "Leaf Mold",
    "Tomato___Septoria_leaf_spot": "Septoria Leaf Spot",
    "Tomato___healthy": "Healthy",
}

ARTIFACT_DIR = "/Volumes/tomato_data/default/raw/tomato/checkpoints/"
os.makedirs(ARTIFACT_DIR, exist_ok=True)

client = MlflowClient()
print(f"Model URI: {MODEL_URI}")

In [0]:
champion_version = client.get_model_version_by_alias(MODEL_NAME, CHAMPION_ALIAS)
print(f"@champion → v{champion_version.version}")
print(f"  run_id: {champion_version.run_id}")
print(f"  status: {champion_version.status}")

In [0]:
# Load the register model
print(f"Loading {MODEL_URI}...")
registered_model = mlflow.pyfunc.load_model(MODEL_URI)
print(f"✓ Loaded: {type(registered_model)}")

In [0]:
# Preprocessing pipeline
eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# For visualization we also keep the "raw" resize without normalization
display_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
])

# Classes come from the test folder structure (ImageFolder sorts alphabetically)
# We use test_dataset.classes to guarantee the same ordering the model was trained on
test_dataset = datasets.ImageFolder(TEST_DIR, transform=eval_transforms)
CLASS_NAMES = test_dataset.classes
print(f"Classes ({len(CLASS_NAMES)}): {CLASS_NAMES}")

In [0]:
# Batch inference accuracy check
# Take a deterministic subset of the test set
rng = np.random.default_rng(42)
indices = rng.choice(len(test_dataset), size=min(EVAL_SUBSET_SIZE, len(test_dataset)), replace=False)
subset = Subset(test_dataset, indices)
subset_loader = DataLoader(subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

all_preds = []
all_labels = []

for batch_imgs, batch_labels in subset_loader:
    # MLflow PyFunc expects numpy. Convert batched tensors to numpy.
    imgs_np = batch_imgs.numpy().astype(np.float32)

    # predict returns class indices for classification models
    preds = registered_model.predict(imgs_np)

    # Handle both shapes: scalar labels or one-hot / probability vectors
    preds = np.asarray(preds)
    if preds.ndim > 1:
        preds = np.argmax(preds, axis=1)

    all_preds.extend(preds.tolist())
    all_labels.extend(batch_labels.numpy().tolist())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

subset_acc = (all_preds == all_labels).mean()
print(f"Subset size: {len(all_labels)}")
print(f"Subset accuracy (from registry): {subset_acc:.4f}")
print(f"(Training-time test accuracy was 0.9141)")

In [0]:
# Confusion matrix from the registered model
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=[CLASS_DISPLAY_NAMES.get(c, c) for c in CLASS_NAMES],
    yticklabels=[CLASS_DISPLAY_NAMES.get(c, c) for c in CLASS_NAMES],
    ax=ax,
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title(f"Confusion Matrix (registry model) — Subset Acc {subset_acc:.3f}")
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
cm_path = os.path.join(ARTIFACT_DIR, "registry_confusion_matrix.png")
plt.savefig(cm_path, dpi=120)
plt.show()
print(f"Saved: {cm_path}")

# Also print the classification report for the subset
report = classification_report(
    all_labels, all_preds,
    target_names=[CLASS_DISPLAY_NAMES.get(c, c) for c in CLASS_NAMES],
    digits=4,
)
print("\nClassification report (registry model):")
print(report)

In [0]:
# Visual grid: one sample per class
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, class_name in enumerate(CLASS_NAMES):
    class_dir = Path(TEST_DIR) / class_name
    sample_path = next(class_dir.glob("*.[jJ][pP][gG]"))

    with Image.open(sample_path) as im:
        im = im.convert("RGB")
        display_img = display_transforms(im)
        tensor = eval_transforms(im).unsqueeze(0).numpy().astype(np.float32)

    preds = registered_model.predict(tensor)
    preds = np.asarray(preds)
    if preds.ndim > 1:
        pred_idx = int(np.argmax(preds[0]))
        probs = preds[0]
    else:
        pred_idx = int(preds[0])
        probs = None

    pred_label = CLASS_DISPLAY_NAMES.get(CLASS_NAMES[pred_idx], CLASS_NAMES[pred_idx])
    true_label = CLASS_DISPLAY_NAMES.get(class_name, class_name)
    correct = (pred_label == true_label)

    ax = axes[idx]
    ax.imshow(display_img)
    ax.axis("off")
    confidence = f"{probs[pred_idx]:.3f}" if probs is not None else "n/a"
    title_color = "green" if correct else "red"
    ax.set_title(
        f"True: {true_label}\nPred: {pred_label} ({confidence})",
        color=title_color,
        fontsize=10,
    )

plt.suptitle("Registry model — one sample per class", fontsize=14)
plt.tight_layout()
grid_path = os.path.join(ARTIFACT_DIR, "registry_sample_grid.png")
plt.savefig(grid_path, dpi=120)
plt.show()
print(f"Saved: {grid_path}")

In [0]:
# Log validation results back to the MLflow run
with mlflow.start_run(run_id=champion_version.run_id):
    mlflow.log_metric("registry_subset_accuracy", float(subset_acc))
    mlflow.log_metric("registry_subset_size", int(len(all_labels)))
    mlflow.log_artifact(cm_path)
    mlflow.log_artifact(grid_path)

print(f"✓ Validation metrics logged to run {champion_version.run_id}")

In [0]:
# Save the class mapping for the Streamlit app
# Write the class mapping the Streamlit app will use
mapping = {
    "class_names": CLASS_NAMES,
    "class_to_idx": test_dataset.class_to_idx,
    "display_names": CLASS_DISPLAY_NAMES,
    "model_uri": MODEL_URI,
    "image_size": IMAGE_SIZE,
    "imagenet_mean": IMAGENET_MEAN,
    "imagenet_std": IMAGENET_STD,
}

mapping_path = os.path.join(ARTIFACT_DIR, "streamlit_class_mapping.json")
with open(mapping_path, "w") as f:
    json.dump(mapping, f, indent=2)

print(f"Saved: {mapping_path}")
print(json.dumps(mapping, indent=2))